Learning goals:
1. Set up a coupled flow and transport 2d simulation.
2. Import of fractures from a file
3. Mesh construction
4. Run simulation
5. 

# Coupled flow and transport in a fractured porous medium
Here we show how to set up a simulation of coupled flow and transport in a fractured porous media using the multiphysics simulation framework in PorePy. The tutorial covers how to specify:
* The geometry of the fracture network and the computational domain;
* Parameters for mesh size control;
* Parameters for permeability; porosity etc.;
* Boundary conditions;
* Simulation time etc.

Although the tutorial in one sense aims to be self-contained, we will make frequent references to other tutorials in PorePy proper, where more details are provided. For instance, it may be advisable to read through the PorePy tutorial on single phase flow before reading further here.

A word of caution: The multiphysics models in PorePy make heavy use of Python mixins (for readers familiar with object oriented programming, mixins can crudely be thought of as inheritance). While this is a powerful technology that allows for code reuse and flexibility in simulation setup, it will lead to unexpected behavior (typically the calling of the wrong version of a polymorphic method). The below code is safe to use, but before composing simulation classes from scratch, it is highly advisable to read up on mixins - the tutorial on single phase flow contains an excellent reference in that regard.

# Specifying a simulation
We start by importing PorePy, as well as numpy and the Path class from pathlib

In [1]:
import porepy as pp
import numpy as np
from pathlib import Path

## Geometry
We will specify the simulation setup through a series of mixin classes. First, we import the fracture network geometry from a csv file and also define the domain.

Points to note:
* How to read fractures and a domain from a CSV file and feed them into the the multiphysics model.
* More generally, which methods to override to set fractures and domain.

In [ ]:
class Geometry:

    def set_fractures(self):
        # The first line in the csv file specifies the domain, so we skip it when
        # reading the fracture data.
        data = np.genfromtxt(Path('fractures.csv'), delimiter=',', skip_header=1)
        fractures = []
        for row in data:
            f = pp.LineFracture(data.reshape((2, -1), order="F"))
                
            fractures.append(f)
        self._fractures = fractures

    def set_domain(self):
        # Read only the first line of the csv file.
        data = np.genfromtxt(Path('fractures.csv'), delimiter=',', max_rows=1)
        self._domain = pp.Domain({"xmin": data[0], "ymin": data[1], "xmax": data[2], "ymax": data[3]})

    

## Boundary conditions
By default, PorePy assign no-flow (homogeneous Neumann) conditions for flow and transport. We therefore only need to specify the conditions along boundaries where other conditions apply. In our case, we will (arbitrarily) set the pressure at the left and right boundaries to 2MPa and 1MPa, respectively, and leave the top and bottom boundaries untouched.

Points to note:
* The type and numerical values for boundary conditions are set by overriding two different methods.
* By default, boundary conditions will be of Neumann type.
* By default, boundary conditions will have the numerical value zero (independent of the type of the condition).
* Setting boundary conditions for mass flow requires some care - see the flow chart (!) in the PorePy tutorial on boundary conditions.

In [ ]:
class BoundaryConditions:

    def bc_type_darcy_flux(self, sd: pp.Grid) -> pp.BoundaryCondition:
        """Boundary condition type (not values!) for Darcy flux.
        """
        # Get hold of the object that describes the sides of the domain. This has
        # attributes north, south, east, and west, which can be used to index faces
        # at the respective sides of the domain.
        domain_sides = self.domain_boundary_sides(sd)
        # Define boundary condition on faces. Dirichlet conditions on the east and west
        # sides - the north and south will be assigned Neumann conditions by default.
        return pp.BoundaryCondition(sd, domain_sides.west + domain_sides.east, "dir")

    def bc_values_pressure(self, bg: pp.BoundaryGrid) -> np.ndarray:
        """Boundary condition values for Darcy flux.
        """
        # Fetch the sides of the domain.
        domain_sides = self.domain_boundary_sides(bg)
        # The return value is an array of zeros with the size of the number of cells in
        # the boundary grid (which is equivalent to the number of boundary faces in the
        # original grid). 
        values = np.zeros(bg.num_cells)
        # The multiphysics models have a system for automatic unit conversion, which can
        # be set up at the configuration step (below). It is good practice to
        # consistently use the conversion system, as in the code below.
        right_value = self.units.convert(1 * pp.MEGA * pp.PASCAL, "Pa")
        left_value = 2 * right_value
        # Set values, return.
        values[domain_sides.right] = right_value
        values[domain_sides.left] = left_value
        return values

    def bc_type_fluid_flux(self, sd: pp.Grid) -> pp.BoundaryCondition:
        """Boundary condition type for the density-mobility product.
        """
        domain_sides = self.domain_boundary_sides(sd)
        return pp.BoundaryCondition(sd, domain_sides.west + domain_sides.east, "dir")

    def bc_values_overall_fraction(
        self, component: pp.Component, bg: pp.BoundaryGrid
    ) -> np.ndarray:
        """Define non-trivial inflow of the tracer component on the inlet (west)."""

        z = np.zeros(bg.num_cells)

        assert component.name == "tracer", "Only the tracer is independent."

        # Set the tracer concentration to 0.5
        domain_sides = self.domain_boundary_sides(bg)
        # Strictly speaking, we need not convert a dimensionless quantity, but we do it
        # for the sake of consistency.
        z[domain_sides.west] = self.units.convert(0.5, "-")
        return z
